# Árboles de Decisión: Regresión y Clasificación
## Ejemplos del PDF + Ejercicio con datos reales

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.tree import (DecisionTreeRegressor, DecisionTreeClassifier,
                          plot_tree, export_text)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
print("Librerías cargadas correctamente ✓")

---
# PARTE 1: Árbol de Regresión — Precio de casas

Reproducción completa del ejemplo del PDF.
**Datos:** 10 casas con superficie (m²), habitaciones y precio (miles USD).
**Árbol:** profundidad máxima = 2.

### 1.1 Dataset

In [ ]:
data_reg = {
    'casa':         list(range(1, 11)),
    'superficie':   [60, 75, 80, 95, 100, 110, 120, 130, 140, 160],
    'habitaciones': [2,  2,  3,  3,  3,   4,   4,   4,   5,   5 ],
    'precio':       [120,145,160,175,190, 220, 240, 255, 280, 310],
}
df_reg = pd.DataFrame(data_reg)
y_bar  = df_reg['precio'].mean()
TSS    = ((df_reg['precio'] - y_bar)**2).sum()

print('Dataset de casas:')
print(df_reg.to_string(index=False))
print(f'\nMedia global: ȳ = {y_bar:.1f}  |  TSS = {TSS:.1f}')

### 1.2 Evaluación manual de todos los umbrales de `superficie`

Calculamos la reducción de RSS para cada umbral candidato, reproduciendo la tabla del PDF.

In [ ]:
y = df_reg['precio'].values
x1 = df_reg['superficie'].values
umbrales = [(x1[i] + x1[i+1]) / 2 for i in range(len(x1)-1)]

def rss(y_arr):
    return ((y_arr - y_arr.mean())**2).sum() if len(y_arr) > 0 else 0.0

print(f'{'Umbral':>9} {'n_izq':>6} {'ȳ_izq':>7} {'RSS_izq':>9} '
      f'{'ȳ_der':>7} {'RSS_der':>9} {'RSS_tot':>9} {'ΔRSS':>9}')
print('-'*75)
best = {'tau': None, 'rtot': 1e9, 'delta': -1}
for tau in umbrales:
    mask = x1 <= tau
    yi, yd = y[mask], y[~mask]
    ri, rd  = rss(yi), rss(yd)
    rtot    = ri + rd
    delta   = TSS - rtot
    flag    = '  <-- MEJOR' if rtot < best['rtot'] else ''
    if rtot < best['rtot']:
        best = {'tau': tau, 'rtot': rtot, 'delta': delta,
                'yi_mean': yi.mean(), 'yd_mean': yd.mean()}
    print(f'{tau:>9.1f} {mask.sum():>6} {yi.mean():>7.1f} {ri:>9.1f} '
          f'{yd.mean():>7.1f} {rd:>9.1f} {rtot:>9.1f} {delta:>9.1f}{flag}')
print(f"\nMejor primera división: x1 <= {best['tau']}  ΔRSS = {best['delta']:.1f}")
print('PDF reporta: x1 <= 97.5 con ΔRSS ≈ 28 867 ✓')

### 1.3 Ajuste con scikit-learn (profundidad = 2)

In [ ]:
X_reg = df_reg[['superficie', 'habitaciones']]
y_reg = df_reg['precio']

tree_reg = DecisionTreeRegressor(max_depth=2, random_state=42)
tree_reg.fit(X_reg, y_reg)
y_pred_reg = tree_reg.predict(X_reg)

print('Estructura del árbol:')
print(export_text(tree_reg, feature_names=['superficie','habitaciones']))

df_reg['hoja']        = tree_reg.apply(X_reg)
df_reg['precio_pred'] = y_pred_reg.round(1)
df_reg['error_abs']   = (y_reg - y_pred_reg).abs().round(1)
print('Predicciones:')
print(df_reg[['casa','superficie','habitaciones',
              'precio','hoja','precio_pred','error_abs']].to_string(index=False))

### 1.4 Métricas completas

In [ ]:
RSS_m  = ((y_reg - y_pred_reg)**2).sum()
ESS_m  = TSS - RSS_m
R2     = 1 - RSS_m / TSS
RMSE   = np.sqrt(mean_squared_error(y_reg, y_pred_reg))
MAE    = mean_absolute_error(y_reg, y_pred_reg)

print('='*52)
print('  MÉTRICAS — Árbol de Regresión (prof. 2)')
print('='*52)
print(f'  TSS (varianza total)     : {TSS:>10.2f}')
print(f'  ESS (varianza explicada) : {ESS_m:>10.2f}')
print(f'  RSS (varianza residual)  : {RSS_m:>10.2f}')
print(f'  R²                       : {R2:>10.4f}  ({R2*100:.1f}%)')
print(f'  RMSE                     : {RMSE:>10.4f}  miles USD')
print(f'  MAE                      : {MAE:>10.4f}  miles USD')
print('='*52)
print(f'\n  PDF: R² = 0.870, MAE = 18.0  |  sklearn: R² = {R2:.3f}, MAE = {MAE:.1f}')

### 1.5 Visualización

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel 1: árbol
ax1 = fig.add_subplot(gs[0, :2])
plot_tree(tree_reg, feature_names=['Superficie (m²)','Habitaciones'],
          filled=True, rounded=True, fontsize=10, ax=ax1, precision=1)
ax1.set_title('Árbol de Regresión — Precio de casas', fontweight='bold')

# Panel 2: real vs predicho
ax2 = fig.add_subplot(gs[0, 2])
ax2.scatter(y_reg, y_pred_reg, s=100, color='steelblue', edgecolors='white', zorder=3)
for _, r in df_reg.iterrows():
    ax2.annotate(f"C{int(r['casa'])}", (r['precio'], r['precio_pred']),
                 textcoords='offset points', xytext=(4,3), fontsize=8)
lim = [110, 320]
ax2.plot(lim, lim, 'r--', lw=1.5, label='Pred. perfecta')
ax2.set_xlabel('Precio real'); ax2.set_ylabel('Precio predicho')
ax2.set_title(f'Real vs Predicho  R²={R2:.3f}', fontweight='bold')
ax2.legend(fontsize=9)

# Panel 3: superficie vs precio (coloreado por hoja)
ax3 = fig.add_subplot(gs[1, 0])
sc = ax3.scatter(df_reg['superficie'], df_reg['precio'],
                 c=df_reg['hoja'], cmap='Set1', s=120,
                 edgecolors='black', zorder=3)
for th in tree_reg.tree_.threshold:
    if 40 < th < 200:
        ax3.axvline(th, color='gray', ls='--', lw=1.2, alpha=0.7)
ax3.set_xlabel('Superficie (m²)'); ax3.set_ylabel('Precio (miles USD)')
ax3.set_title('Regiones del árbol', fontweight='bold')
plt.colorbar(sc, ax=ax3, label='Hoja')

# Panel 4: residuos
ax4 = fig.add_subplot(gs[1, 1])
res = y_reg.values - y_pred_reg
ax4.bar(df_reg['casa'], res,
        color=['steelblue' if r >= 0 else 'crimson' for r in res], edgecolor='white')
ax4.axhline(0, color='black', lw=1.5)
ax4.set_xlabel('Casa'); ax4.set_ylabel('Residuo')
ax4.set_title(f'Residuos  MAE={MAE:.1f} miles USD', fontweight='bold')
ax4.set_xticks(df_reg['casa'])

# Panel 5: TSS = ESS + RSS
ax5 = fig.add_subplot(gs[1, 2])
vals = [TSS, ESS_m, RSS_m]
cols = ['#2196F3','#4CAF50','#F44336']
cats = ['TSS', 'ESS', 'RSS']
bars = ax5.bar(cats, vals, color=cols, edgecolor='white', width=0.5)
for b, v in zip(bars, vals):
    ax5.text(b.get_x()+b.get_width()/2, v+150, f'{v:.0f}',
             ha='center', fontweight='bold', fontsize=10)
ax5.set_ylabel('Suma de cuadrados')
ax5.set_title(f'TSS = ESS + RSS  |  R²={R2:.3f}', fontweight='bold')

fig.suptitle('ÁRBOL DE REGRESIÓN — Precio de casas', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print("Visualización generada ✓")

### 1.6 Importancia de variables

In [ ]:
imp_reg = pd.Series(tree_reg.feature_importances_,
                    index=['Superficie (m²)', 'Habitaciones']).sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(imp_reg.index, imp_reg.values,
        color=['#2196F3','#FF9800'], edgecolor='white', height=0.4)
for i, v in enumerate(imp_reg.values):
    ax.text(v+0.005, i, f'{v:.4f}', va='center', fontweight='bold')
ax.set_xlabel('Reducción relativa de RSS')
ax.set_title('Importancia de variables — Regresión', fontweight='bold')
ax.set_xlim(0, 1.15)
plt.tight_layout(); plt.show()
print(f'Variable más importante: {imp_reg.idxmax()}  ({imp_reg.max():.4f})')

---
# PARTE 2: Árbol de Clasificación — Diagnóstico cardíaco

Reproducción completa del ejemplo del PDF.
**Datos:** 12 pacientes con colesterol (mg/dL) y presión sistólica (mmHg).
**Respuesta:** `y ∈ {0=sano, 1=enfermo}`.  **Árbol:** profundidad = 2.

### 2.1 Dataset

In [ ]:
data_cls = {
    'paciente':   list(range(1,13)),
    'colesterol': [180,195,210,215,220,230,235,240,245,250,260,270],
    'presion':    [120,125,130,135,140,130,145,150,128,155,160,135],
    'enfermedad': [0,  0,  0,  1,  1,  0,  1,  1,  0,  1,  1,  0 ],
}
df_cls = pd.DataFrame(data_cls)
df_cls['clase'] = df_cls['enfermedad'].map({0:'Sano (−)', 1:'Enfermo (+)'})
print(df_cls[['paciente','colesterol','presion','clase']].to_string(index=False))
p_pos = df_cls['enfermedad'].mean()
G_root = 2 * p_pos * (1 - p_pos)
print(f'\nGini raíz = 2×{p_pos:.2f}×{1-p_pos:.2f} = {G_root:.4f}')

### 2.2 Evaluación manual de umbrales — Índice de Gini

Reproducimos exactamente la tabla del PDF para `colesterol` y `presión`.

In [ ]:
def gini(y_arr):
    if len(y_arr) == 0: return 0.0
    p = y_arr.mean()
    return 2 * p * (1 - p)

def gini_pond(y, mask):
    n = len(y)
    yi, yd = y[mask], y[~mask]
    if len(yi)==0 or len(yd)==0: return np.inf
    return len(yi)/n * gini(yi) + len(yd)/n * gini(yd)

y_c = df_cls['enfermedad'].values
G0  = gini(y_c)

for feat in ['colesterol', 'presion']:
    x  = df_cls[feat].values
    xu = np.sort(np.unique(x))
    taus = [(xu[i]+xu[i+1])/2 for i in range(len(xu)-1)]
    print(f'\n{'─'*75}')
    print(f'  Variable: {feat}')
    print(f'{'─'*75}')
    print(f'  {'Umbral':>8} {'n_izq':>6} {'p+_izq':>8} {'G_izq':>7} '
          f'{'n_der':>6} {'p+_der':>8} {'G_der':>7} {'G_pond':>8} {'ΔG':>8}')
    best_g, best_t = np.inf, None
    for tau in taus:
        mask   = x <= tau
        yi, yd = y_c[mask], y_c[~mask]
        gi, gd = gini(yi), gini(yd)
        gp     = gini_pond(y_c, mask)
        dg     = G0 - gp
        flag   = '  <--' if gp < best_g else ''
        if gp < best_g: best_g, best_t = gp, tau
        print(f'  {tau:>8.1f} {mask.sum():>6} {yi.mean():>8.3f} {gi:>7.4f} '
              f'{(~mask).sum():>6} {yd.mean():>8.3f} {gd:>7.4f} {gp:>8.4f} {dg:>8.4f}{flag}')
    print(f'  => Mejor: {feat} <= {best_t:.1f}  G_pond={best_g:.4f}  ΔG={G0-best_g:.4f}')

### 2.3 Ajuste con scikit-learn

In [ ]:
X_cls = df_cls[['colesterol','presion']]
y_cls = df_cls['enfermedad']

tree_cls = DecisionTreeClassifier(max_depth=2, criterion='gini', random_state=42)
tree_cls.fit(X_cls, y_cls)
y_pred_cls = tree_cls.predict(X_cls)

print('Estructura del árbol:')
print(export_text(tree_cls, feature_names=['colesterol','presion']))

df_cls['pred_cls']  = y_pred_cls
df_cls['pred_clase']= pd.Series(y_pred_cls).map({0:'Sano (−)',1:'Enfermo (+)'}).values
df_cls['correcto']  = (y_pred_cls == y_cls.values)
print('Resultados:')
print(df_cls[['paciente','colesterol','presion',
              'clase','pred_clase','correcto']].to_string(index=False))

### 2.4 Métricas de clasificación

In [ ]:
acc   = accuracy_score(y_cls, y_pred_cls)
prec  = precision_score(y_cls, y_pred_cls, zero_division=0)
rec   = recall_score(y_cls, y_pred_cls, zero_division=0)
f1    = f1_score(y_cls, y_pred_cls, zero_division=0)
cm    = confusion_matrix(y_cls, y_pred_cls)
VN,FP,FN,VP = cm[0,0],cm[0,1],cm[1,0],cm[1,1]

print('='*52)
print('  MÉTRICAS — Árbol de Clasificación (prof. 2)')
print('='*52)
print(f'  Matriz de confusión:')
print(f'                   Pred. Sano  Pred. Enf.')
print(f'  Real Sano            {VN:>4}       {FP:>4}')
print(f'  Real Enfermo         {FN:>4}       {VP:>4}')
print(f'  {"-"*44}')
print(f'  Accuracy    : {acc:.4f}  ({acc*100:.1f}%)')
print(f'  Precisión   : {prec:.4f}')
print(f'  Recall      : {rec:.4f}')
print(f'  F1-score    : {f1:.4f}')
print('='*52)
print(f'PDF: accuracy = 11/12 = 91.7%  =>  {acc*100:.1f}% ✓')

### 2.5 Visualización

In [ ]:
fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

# Panel 1: árbol
ax1 = fig.add_subplot(gs[0, :2])
plot_tree(tree_cls, feature_names=['Colesterol','Presión'],
          class_names=['Sano (−)','Enfermo (+)'],
          filled=True, rounded=True, fontsize=10, ax=ax1, precision=4)
ax1.set_title('Árbol de Clasificación — Diagnóstico cardíaco', fontweight='bold')

# Panel 2: matriz de confusión
ax2 = fig.add_subplot(gs[0, 2])
ConfusionMatrixDisplay(cm, display_labels=['Sano','Enfermo']).plot(
    ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title(f'Matriz de confusión  Acc={acc:.1%}', fontweight='bold')

# Panel 3: frontera de decisión
ax3 = fig.add_subplot(gs[1, 0])
xx1, xx2 = np.meshgrid(np.linspace(170,280,300), np.linspace(115,170,300))
Z = tree_cls.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)
ax3.contourf(xx1, xx2, Z, alpha=0.22, cmap='RdYlGn_r', levels=[-0.5,0.5,1.5])
col_map = {0:'green', 1:'red'}
for _, row in df_cls.iterrows():
    mk = 'o' if row['correcto'] else 'X'
    ax3.scatter(row['colesterol'], row['presion'], color=col_map[row['enfermedad']],
                marker=mk, s=130, edgecolors='black', lw=1.2, zorder=5)
    ax3.annotate(f"P{int(row['paciente'])}",
                 (row['colesterol'], row['presion']),
                 textcoords='offset points', xytext=(4,4), fontsize=8)
ax3.set_xlabel('Colesterol (mg/dL)'); ax3.set_ylabel('Presión (mmHg)')
ax3.set_title('Frontera de decisión', fontweight='bold')

# Panel 4: importancia
ax4 = fig.add_subplot(gs[1, 1])
imp_cls = pd.Series(tree_cls.feature_importances_,
                    index=['Colesterol','Presión']).sort_values()
imp_cls.plot(kind='barh', ax=ax4, color=['#FF9800','#2196F3'], edgecolor='white')
for i, v in enumerate(imp_cls.values):
    ax4.text(v+0.005, i, f'{v:.4f}', va='center', fontweight='bold')
ax4.set_xlabel('Reducción de Gini')
ax4.set_title('Importancia de variables', fontweight='bold')
ax4.set_xlim(0, 1.1)

# Panel 5: Gini vs umbral (presión)
ax5 = fig.add_subplot(gs[1, 2])
pres_u = np.sort(np.unique(df_cls['presion'].values))
taus_p = [(pres_u[i]+pres_u[i+1])/2 for i in range(len(pres_u)-1)]
gini_vals = [gini_pond(y_c, df_cls['presion'].values<=t) for t in taus_p]
ax5.plot(taus_p, gini_vals, 'o-', color='steelblue', lw=2, ms=8)
bi = np.argmin(gini_vals)
ax5.scatter([taus_p[bi]], [gini_vals[bi]], s=200, color='crimson', zorder=5,
            label=f'Mejor τ={taus_p[bi]:.1f}\nGini={gini_vals[bi]:.3f}')
ax5.axhline(G0, color='gray', ls='--', label=f'Gini raíz={G0:.3f}')
ax5.set_xlabel('Umbral presión (mmHg)'); ax5.set_ylabel('Gini ponderado')
ax5.set_title('Gini vs umbral — Presión', fontweight='bold')
ax5.legend(fontsize=8)

fig.suptitle('ÁRBOL DE CLASIFICACIÓN — Diagnóstico cardíaco',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()
print("Visualización generada ✓")

---
# PARTE 3: Ejercicio para Estudiantes

## Datasets reales

### Dataset A — `tips` → Árbol de **Regresión**
Propinas en un restaurante estadounidense (244 registros, Bryant & Smith, 1995).
- **Descarga:** `seaborn.load_dataset('tips')` (datos en GitHub de Seaborn).
- **Target:** `tip` — propina en dólares.
- **Predictores:** `total_bill`, `size`, `sex`, `smoker`, `day`, `time`.

### Dataset B — `penguins` → Árbol de **Clasificación**
Pingüinos del archipiélago Palmer, Antártida (333 registros, Gorman et al., *PLOS ONE*, 2014).
- **Descarga:** `seaborn.load_dataset('penguins')` (datos en GitHub de Seaborn).
- **Target:** `species` — Adelie / Chinstrap / Gentoo (3 clases).
- **Predictores:** `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`,
  `body_mass_g`, `island`, `sex`.

---
> **Instrucción:**  
> Las celdas marcadas con 🔴 deben completarse.
> Las marcadas con ✅ son código de apoyo ya resuelto.

### 3.0 Carga de datos (✅ resuelto)

In [ ]:
tips     = sns.load_dataset('tips')
penguins = sns.load_dataset('penguins').dropna().reset_index(drop=True)

print('─'*55)
print('DATASET TIPS (regresión)')
print('─'*55)
print(f'Forma: {tips.shape}  |  columnas: {list(tips.columns)}')
print(tips.head())
print(f'\nEstadísticas de tip:')
print(tips['tip'].describe().round(2))

print('\n' + '─'*55)
print('DATASET PENGUINS (clasificación)')
print('─'*55)
print(f'Forma: {penguins.shape}  |  columnas: {list(penguins.columns)}')
print(penguins.head())
print('\nDistribución de especies:')
print(penguins['species'].value_counts())

### 3.1 Análisis exploratorio (✅ resuelto)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Análisis Exploratorio', fontsize=14, fontweight='bold')

# Tips: distribución del target
axes[0,0].hist(tips['tip'], bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].axvline(tips['tip'].mean(), color='crimson', lw=2,
                  label=f"Media={tips['tip'].mean():.2f}")
axes[0,0].set_xlabel('Propina (USD)')
axes[0,0].set_title('Tips: distribución de propinas', fontweight='bold')
axes[0,0].legend()

# Tips: propina vs cuenta total
axes[0,1].scatter(tips['total_bill'], tips['tip'], alpha=0.5,
                  color='steelblue', edgecolors='white', s=60)
axes[0,1].set_xlabel('Cuenta total (USD)'); axes[0,1].set_ylabel('Propina (USD)')
axes[0,1].set_title('Tips: propina vs cuenta total', fontweight='bold')

# Tips: boxplot por día
tips.boxplot(column='tip', by='day', ax=axes[0,2])
axes[0,2].set_xlabel('Día'); axes[0,2].set_ylabel('Propina (USD)')
plt.sca(axes[0,2]); plt.title('Tips: propina por día', fontweight='bold')

# Penguins: pico vs aleta
colores_e = {'Adelie':'#1f77b4','Chinstrap':'#ff7f0e','Gentoo':'#2ca02c'}
for esp, col in colores_e.items():
    sub = penguins[penguins['species']==esp]
    axes[1,0].scatter(sub['bill_length_mm'], sub['flipper_length_mm'],
                      label=esp, color=col, alpha=0.7, edgecolors='white', s=60)
axes[1,0].set_xlabel('Longitud pico (mm)'); axes[1,0].set_ylabel('Longitud aleta (mm)')
axes[1,0].set_title('Penguins: pico vs aleta', fontweight='bold')
axes[1,0].legend(fontsize=8)

# Penguins: frecuencia por isla
pd.crosstab(penguins['species'], penguins['island']).plot(
    kind='bar', ax=axes[1,1], edgecolor='white', alpha=0.85)
axes[1,1].set_title('Penguins: especies por isla', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=0)
axes[1,1].legend(title='Isla', fontsize=8)

# Penguins: masa por especie
penguins.boxplot(column='body_mass_g', by='species', ax=axes[1,2])
axes[1,2].set_xlabel('')
plt.sca(axes[1,2]); plt.title('Penguins: masa corporal por especie', fontweight='bold')

plt.tight_layout(); plt.show()
print("Exploración completada ✓")

### 3.2 Preprocesamiento (✅ resuelto)

In [ ]:
# Tips: codificar categóricas
tips_enc = tips.copy()
for col in ['sex','smoker','day','time']:
    tips_enc[col] = LabelEncoder().fit_transform(tips_enc[col])
X_tips = tips_enc.drop(columns=['tip'])
y_tips = tips_enc['tip']
print('Tips — features:')
print(X_tips.dtypes)

# Penguins: codificar categóricas
penguins_enc = penguins.copy()
le_sp = LabelEncoder()
penguins_enc['species_enc'] = le_sp.fit_transform(penguins_enc['species'])
for col in ['island','sex']:
    penguins_enc[col] = LabelEncoder().fit_transform(penguins_enc[col])
X_peng = penguins_enc.drop(columns=['species','species_enc'])
y_peng = penguins_enc['species_enc']
print(f'\nClases: {list(enumerate(le_sp.classes_))}')
print('Penguins — features:')
print(X_peng.dtypes)

---
## 🌳 Ejercicio A — Árbol de Regresión con `tips`

**Objetivo:** predecir la propina (`tip`) a partir de las características del servicio.

Completa los pasos marcados con 🔴.

#### A1: División train/test (🔴)

In [ ]:
# 🔴 Divide X_tips, y_tips en entrenamiento (80%) y prueba (20%).
# Usa random_state=42. Variables: X_train_r, X_test_r, y_train_r, y_test_r.

# X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
#     X_tips, y_tips, test_size=0.2, random_state=42)
# print(f'Train: {len(X_train_r)} | Test: {len(X_test_r)}')

#### A2: Ajuste del árbol con profundidad 3 (🔴)

In [ ]:
# 🔴 Ajusta un DecisionTreeRegressor con max_depth=3.
# Imprime la estructura con export_text.

# tree_tips = DecisionTreeRegressor(max_depth=3, random_state=42)
# tree_tips.fit(X_train_r, y_train_r)
# print(export_text(tree_tips, feature_names=list(X_tips.columns)))

#### A3: Predicciones y métricas en train y test (🔴)

In [ ]:
# 🔴 Calcula RMSE, MAE y R² en entrenamiento Y prueba.
# Compara ambos para detectar sobreajuste.

# y_pred_train = tree_tips.predict(X_train_r)
# y_pred_test  = tree_tips.predict(X_test_r)

# for split, yt, yp in [('TRAIN', y_train_r, y_pred_train),
#                        ('TEST',  y_test_r,  y_pred_test)]:
#     print(f'{split}:')
#     print(f'  RMSE = {np.sqrt(mean_squared_error(yt, yp)):.4f}')
#     print(f'  MAE  = {mean_absolute_error(yt, yp):.4f}')
#     print(f'  R²   = {r2_score(yt, yp):.4f}')

#### A4: Curva de bias–varianza para selección de profundidad (🔴)

In [ ]:
# 🔴 Itera max_depth de 1 a 10.
# Grafica R² en train y test. Identifica sobreajuste.

depths = range(1, 11)
# r2_train_v, r2_test_v = [], []
# for d in depths:
#     t = DecisionTreeRegressor(max_depth=d, random_state=42)
#     t.fit(X_train_r, y_train_r)
#     r2_train_v.append(r2_score(y_train_r, t.predict(X_train_r)))
#     r2_test_v.append(r2_score(y_test_r,  t.predict(X_test_r)))

# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(depths, r2_train_v, 'o-', label='Train R²', color='steelblue')
# ax.plot(depths, r2_test_v,  's-', label='Test R²',  color='crimson')
# opt_d = depths[np.argmax(r2_test_v)]
# ax.axvline(opt_d, color='gray', ls='--', label=f'Depth óptima={opt_d}')
# ax.set_xlabel('Profundidad'); ax.set_ylabel('R²')
# ax.set_title('Bias–varianza en función de la profundidad — Tips')
# ax.legend(); plt.tight_layout(); plt.show()
# print(f'Profundidad óptima: {opt_d}')

#### A5: Importancia de variables (🔴)

In [ ]:
# 🔴 Con el árbol de profundidad óptima:
# 1. Grafica la importancia de variables.
# 2. ¿Cuál es la variable más importante para predecir la propina?
# 3. ¿Tiene sentido desde el punto de vista práctico?

# imp_tips = pd.Series(tree_tips.feature_importances_,
#                      index=X_tips.columns).sort_values()
# fig, ax = plt.subplots(figsize=(7, 4))
# imp_tips.plot(kind='barh', ax=ax, edgecolor='white')
# ax.set_xlabel('Importancia (reducción de RSS)')
# ax.set_title('Importancia de variables — Tips')
# plt.tight_layout(); plt.show()

---
## 🌳 Ejercicio B — Árbol de Clasificación con `penguins`

**Objetivo:** predecir la especie de pingüino (`species`) a partir de
sus características morfológicas.  
**Importante:** hay **3 clases** (Adelie, Chinstrap, Gentoo).

Completa los pasos marcados con 🔴.

#### B1: División train/test estratificada (🔴)

In [ ]:
# 🔴 Divide X_peng, y_peng en 75% train y 25% test.
# Usa stratify=y_peng para mantener proporción de clases.
# Variables: X_train_c, X_test_c, y_train_c, y_test_c.

# X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
#     X_peng, y_peng, test_size=0.25, random_state=42, stratify=y_peng)
# print(f'Train: {len(X_train_c)} | Test: {len(X_test_c)}')
# print('Distribución en train:', pd.Series(y_train_c).value_counts().to_dict())
# print('Distribución en test: ', pd.Series(y_test_c).value_counts().to_dict())

#### B2: Ajuste del árbol y visualización (🔴)

In [ ]:
# 🔴 Ajusta un DecisionTreeClassifier con max_depth=3 y criterion='gini'.
# Visualiza el árbol con plot_tree (incluye class_names=le_sp.classes_).

# tree_peng = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
# tree_peng.fit(X_train_c, y_train_c)

# fig, ax = plt.subplots(figsize=(20, 8))
# plot_tree(tree_peng, feature_names=list(X_peng.columns),
#           class_names=le_sp.classes_,
#           filled=True, rounded=True, ax=ax, fontsize=9)
# ax.set_title('Árbol de Clasificación — Pingüinos (prof.=3)', fontweight='bold')
# plt.tight_layout(); plt.show()

#### B3: Métricas completas de clasificación multiclase (🔴)

In [ ]:
# 🔴 Calcula sobre el CONJUNTO DE PRUEBA:
#   1. Accuracy
#   2. Reporte completo (precision, recall, F1 por cada especie)
#   3. Matriz de confusión visualizada
# Interpreta: ¿qué especie se clasifica peor y por qué?

# y_pred_c = tree_peng.predict(X_test_c)

# print('Accuracy:', round(accuracy_score(y_test_c, y_pred_c), 4))
# print('\nReporte de clasificación:')
# print(classification_report(y_test_c, y_pred_c,
#                             target_names=le_sp.classes_))

# cm_p = confusion_matrix(y_test_c, y_pred_c)
# fig, ax = plt.subplots(figsize=(6,5))
# ConfusionMatrixDisplay(cm_p, display_labels=le_sp.classes_).plot(
#     ax=ax, colorbar=False, cmap='Blues')
# ax.set_title('Matriz de confusión — Pingüinos')
# plt.tight_layout(); plt.show()

#### B4: Selección de profundidad con validación cruzada 5-fold (🔴)

In [ ]:
# 🔴 Usa cross_val_score (cv=5, scoring='accuracy') para max_depth 1..10.
# Grafica media ± std del accuracy de CV.
# ¿Con qué profundidad se estabiliza el accuracy?

depths = range(1, 11)
# cv_means, cv_stds = [], []
# for d in depths:
#     t = DecisionTreeClassifier(max_depth=d, random_state=42)
#     sc = cross_val_score(t, X_peng, y_peng, cv=5, scoring='accuracy')
#     cv_means.append(sc.mean()); cv_stds.append(sc.std())

# fig, ax = plt.subplots(figsize=(8, 4))
# ax.plot(depths, cv_means, 'o-', color='steelblue', label='CV Accuracy')
# ax.fill_between(depths,
#                 np.array(cv_means)-np.array(cv_stds),
#                 np.array(cv_means)+np.array(cv_stds),
#                 alpha=0.2, color='steelblue', label='±1 std')
# opt = depths[np.argmax(cv_means)]
# ax.axvline(opt, color='crimson', ls='--', label=f'Óptima={opt}')
# ax.set_xlabel('max_depth'); ax.set_ylabel('CV Accuracy')
# ax.set_title('Selección de profundidad (CV 5-fold) — Pingüinos')
# ax.legend(); plt.tight_layout(); plt.show()
# print(f'Profundidad óptima (CV): {opt}')

#### B5: Importancia de variables y frontera de decisión 2D (🔴)

In [ ]:
# 🔴 Con el árbol entrenado:
# 1. Grafica importancia de variables.
# 2. Usa las 2 más importantes para graficar la frontera de decisión.
#    (Entrena un árbol auxiliar solo con esas 2 variables.)

# imp_peng = pd.Series(tree_peng.feature_importances_,
#                      index=X_peng.columns).sort_values(ascending=False)
# print(imp_peng)
# top2 = imp_peng.index[:2].tolist()
# print(f'Top 2 variables: {top2}')

# X2 = penguins_enc[top2]
# tree_2d = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X2, y_peng)

# x1_r = np.linspace(X2.iloc[:,0].min()-1, X2.iloc[:,0].max()+1, 300)
# x2_r = np.linspace(X2.iloc[:,1].min()-1, X2.iloc[:,1].max()+1, 300)
# xx1, xx2 = np.meshgrid(x1_r, x2_r)
# Z = tree_2d.predict(np.c_[xx1.ravel(), xx2.ravel()]).reshape(xx1.shape)
# fig, ax = plt.subplots(figsize=(8, 6))
# ax.contourf(xx1, xx2, Z, alpha=0.3, cmap='Set1')
# for k, (esp, col) in enumerate(colores_e.items()):
#     sub = penguins_enc[y_peng==k]
#     ax.scatter(sub[top2[0]], sub[top2[1]], color=col, label=esp,
#                edgecolors='black', s=60, alpha=0.7)
# ax.set_xlabel(top2[0]); ax.set_ylabel(top2[1])
# ax.set_title('Frontera de decisión 2D — Pingüinos')
# ax.legend(); plt.tight_layout(); plt.show()

---
## 📝 Preguntas de análisis (🔴 responder aquí)

**Sobre el árbol de regresión (tips):**

1. ¿Cuál es la variable más importante para predecir la propina?
   ¿Es consistente con la intuición? ¿Por qué?

2. ¿En qué profundidad empieza el sobreajuste según la curva R² train vs test?
   ¿Cómo se evidencia?

3. ¿Por qué el MAE es una métrica más interpretable que el RMSE en este contexto?

**Sobre el árbol de clasificación (penguins):**

4. ¿Qué especie se clasifica peor? ¿Con cuál se confunde más? ¿Por qué crees que ocurre?

5. ¿Con cuántas variables morfológicas sería suficiente para una clasificación
   precisa de las tres especies? Justifica con la importancia de variables.

6. ¿Por qué es mejor usar validación cruzada que una sola partición train/test
   para elegir `max_depth`? ¿Qué problema resuelve?

### Respuestas

*Escribe aquí tus respuestas.*

**1.** ...

**2.** ...

**3.** ...

**4.** ...

**5.** ...

**6.** ...

---
## 🗺️ Referencia rápida

| Concepto | Regresión | Clasificación |
|---|---|---|
| Predicción en hoja | $\bar{y}_{R_m}$ (media) | clase mayoritaria |
| Criterio split | min RSS | min Gini / max IG |
| Métrica principal | $R^2$, RMSE, MAE | Accuracy, F₁ |
| Sobreajuste | $R^2_{\text{train}} \gg R^2_{\text{test}}$ | Acc$_{\text{train}} \gg$ Acc$_{\text{test}}$ |

$$\text{Gini} = 1 - \sum_k \hat{p}_{mk}^2 \qquad R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$$